# Sentinel Stroke - Test & Visualization Notebook
**Run inference with trained models and visualize results on Google Colab**

This notebook:
1. Installs dependencies & uploads your models
2. Loads all 3 base architectures (nnU-Net, Swin-UNETR, SegResNet) + Fusion
3. Runs evaluation on your pre-computed fusion_data (NPZ files)
4. Produces rich visualizations: overlays, per-model comparison, metrics dashboard

## 1. Setup & Install Dependencies

In [ ]:
# Install required packages
!pip install -q torch torchvision monai[all] nibabel SimpleITK numpy scipy scikit-image matplotlib seaborn pandas tqdm

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

## 2. Upload Project Files

**Option A:** Mount Google Drive (if you uploaded the project there)  
**Option B:** Upload a zip of the project directly

In [ ]:
# === OPTION A: Mount Google Drive ===
# Uncomment these lines if your project is on Google Drive

from google.colab import drive
drive.mount('/content/drive')

# Set this to your project path on Drive
PROJECT_DIR = '/content/drive/MyDrive/stroke_mvp'

# === OPTION B: Upload zip ===
# Uncomment these lines to upload a zip file instead
#
# from google.colab import files
# uploaded = files.upload()  # Upload stroke_mvp.zip
# !unzip -q stroke_mvp.zip -d /content/
# PROJECT_DIR = '/content/stroke_mvp'

import sys, os
sys.path.insert(0, PROJECT_DIR)
os.chdir(PROJECT_DIR)
print(f"Working directory: {os.getcwd()}")
print(f"Contents: {os.listdir('.')}")

## 3. Load All Models

In [ ]:
import torch
import numpy as np
from pathlib import Path

from src.models.segresnet import SegResNetWrapper
from src.models.swin_unetr import SwinUNETRWrapper
from src.models.fusion_network import AdaptiveFusionNetwork

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# ─── Load SegResNet (5 folds) ───
segresnets = []
for fold in range(5):
    ckpt = Path(f'trained_models/segresnet/fold_{fold}/checkpoint_best.pth')
    if ckpt.exists():
        model, _ = SegResNetWrapper.load_checkpoint(ckpt, device=device)
        model.eval()
        segresnets.append(model)
        print(f"  SegResNet fold {fold} loaded")
print(f"Total SegResNet folds: {len(segresnets)}")

# ─── Load Swin-UNETR (5 folds) ───
swin_unetrs = []
for fold in range(5):
    ckpt = Path(f'trained_models/swin_unetr/fold_{fold}/checkpoint_best.pth')
    if ckpt.exists():
        model, _ = SwinUNETRWrapper.load_checkpoint(ckpt, device=device)
        model.eval()
        swin_unetrs.append(model)
        print(f"  Swin-UNETR fold {fold} loaded")
print(f"Total Swin-UNETR folds: {len(swin_unetrs)}")

# ─── Load Fusion Network ───
fusion_ckpt = Path('checkpoints/fusion_v6_best.pth')
if not fusion_ckpt.exists():
    fusion_ckpt = Path('trained_models/fusion_network_v4_specialist.pth')

fusion_net = AdaptiveFusionNetwork(in_channels=8, use_residual=True).to(device)
if fusion_ckpt.exists():
    ckpt_data = torch.load(fusion_ckpt, map_location=device, weights_only=False)
    fusion_net.load_state_dict(ckpt_data['model_state_dict'])
    print(f"Fusion network loaded from: {fusion_ckpt}")
fusion_net.eval()

print(f"\nAll models loaded successfully!")

## 4. Load Test Data (fusion_data NPZ files)

In [ ]:
from pathlib import Path
import numpy as np

fusion_data_dir = Path('fusion_data')
npz_files = sorted(list(fusion_data_dir.glob('*.npz')))
print(f"Found {len(npz_files)} test cases in fusion_data/")

# Inspect first case to understand format
sample = dict(np.load(npz_files[0]))
print(f"\nSample case: {npz_files[0].stem}")
print(f"Keys: {list(sample.keys())}")
for k, v in sample.items():
    print(f"  {k}: shape={v.shape}, dtype={v.dtype}, range=[{v.min():.4f}, {v.max():.4f}]")

## 5. Run Evaluation on All Test Cases

In [ ]:
import pandas as pd
from tqdm.notebook import tqdm

def compute_metrics(pred, label):
    """Compute Dice, Sensitivity, Specificity, Precision."""
    pred_flat = pred.flatten().astype(bool)
    label_flat = label.flatten().astype(bool)
    
    tp = np.sum(pred_flat & label_flat)
    fp = np.sum(pred_flat & ~label_flat)
    tn = np.sum(~pred_flat & ~label_flat)
    fn = np.sum(~pred_flat & label_flat)
    
    dice = (2 * tp) / (2 * tp + fp + fn + 1e-8)
    sensitivity = tp / (tp + fn + 1e-8)
    specificity = tn / (tn + fp + 1e-8)
    precision_val = tp / (tp + fp + 1e-8)
    
    return {
        'dice': float(dice),
        'sensitivity': float(sensitivity),
        'specificity': float(specificity),
        'precision': float(precision_val),
        'label_volume': int(np.sum(label_flat)),
        'pred_volume': int(np.sum(pred_flat)),
    }

def categorize_lesion_size(volume_voxels):
    """Categorize lesion size."""
    volume_ml = volume_voxels * 0.001
    if volume_ml < 5:
        return 'small'
    elif volume_ml < 50:
        return 'medium'
    else:
        return 'large'

def run_fusion(data, fusion_model, device):
    """Run learned fusion on pre-computed predictions."""
    predictions = np.stack([
        data.get('nnunet', np.zeros_like(data['label'])),
        data.get('swin_unetr', np.zeros_like(data['label'])),
        data.get('segresnet', np.zeros_like(data['label'])),
    ], axis=0).astype(np.float32)
    
    image = data['image'].astype(np.float32)
    
    pred_tensor = torch.from_numpy(predictions).unsqueeze(0).to(device)
    image_tensor = torch.from_numpy(image).unsqueeze(0).to(device)
    
    with torch.no_grad():
        output = fusion_model(pred_tensor, image_tensor)
        fused_prob = output[0, 0].cpu().numpy()
    
    return fused_prob

# ─── Evaluate all cases ───
results = []

for npz_file in tqdm(npz_files, desc="Evaluating"):
    case_id = npz_file.stem
    data = dict(np.load(npz_file))
    
    if 'label' not in data:
        continue
    
    label = data['label']
    row = {'case_id': case_id}
    row['size_category'] = categorize_lesion_size(label.sum())
    
    # Individual models
    for model_name in ['nnunet', 'swin_unetr', 'segresnet']:
        if model_name in data:
            pred_bin = (data[model_name] > 0.5).astype(np.uint8)
            metrics = compute_metrics(pred_bin, label)
            for k, v in metrics.items():
                row[f'{model_name}_{k}'] = v
    
    # Simple average
    preds_list = [data[m] for m in ['nnunet', 'swin_unetr', 'segresnet'] if m in data]
    if preds_list:
        avg_pred = np.mean(preds_list, axis=0)
        avg_bin = (avg_pred > 0.5).astype(np.uint8)
        metrics = compute_metrics(avg_bin, label)
        for k, v in metrics.items():
            row[f'simple_avg_{k}'] = v
    
    # Learned fusion
    if 'image' in data:
        fused_prob = run_fusion(data, fusion_net, device)
        fused_bin = (fused_prob > 0.5).astype(np.uint8)
        metrics = compute_metrics(fused_bin, label)
        for k, v in metrics.items():
            row[f'learned_fusion_{k}'] = v
    
    results.append(row)

df = pd.DataFrame(results)
print(f"\nEvaluated {len(df)} cases")
df.head()

## 6. Metrics Summary Dashboard

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid', font_scale=1.2)

# ─── Overall Dice Comparison ───
methods = ['nnunet', 'swin_unetr', 'segresnet', 'simple_avg', 'learned_fusion']
method_labels = ['nnU-Net', 'Swin-UNETR', 'SegResNet', 'Simple Avg', 'Learned Fusion']
colors = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12', '#9b59b6']

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

for idx, metric in enumerate(['dice', 'sensitivity', 'precision']):
    ax = axes[idx]
    data_for_plot = []
    labels_for_plot = []
    colors_for_plot = []
    
    for m, label, color in zip(methods, method_labels, colors):
        col = f'{m}_{metric}'
        if col in df.columns:
            data_for_plot.append(df[col].dropna().values)
            labels_for_plot.append(label)
            colors_for_plot.append(color)
    
    bp = ax.boxplot(data_for_plot, labels=labels_for_plot, patch_artist=True,
                    medianprops=dict(color='black', linewidth=2))
    
    for patch, color in zip(bp['boxes'], colors_for_plot):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    
    ax.set_title(metric.capitalize(), fontsize=14, fontweight='bold')
    ax.set_ylabel('Score')
    ax.tick_params(axis='x', rotation=30)
    ax.set_ylim(0, 1.05)

fig.suptitle('Model Performance Comparison', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('metrics_comparison.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

# ─── Print summary table ───
print("\n" + "="*70)
print(f"{'Method':<20} {'Dice':>10} {'Sensitivity':>12} {'Precision':>10}")
print("="*70)
for m, label in zip(methods, method_labels):
    dice_col = f'{m}_dice'
    if dice_col in df.columns:
        d = df[dice_col].mean()
        s = df[f'{m}_sensitivity'].mean() if f'{m}_sensitivity' in df.columns else 0
        p = df[f'{m}_precision'].mean() if f'{m}_precision' in df.columns else 0
        print(f"{label:<20} {d:>10.4f} {s:>12.4f} {p:>10.4f}")
print("="*70)

## 7. Stratified Analysis by Lesion Size

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
size_order = ['small', 'medium', 'large']

for idx, size in enumerate(size_order):
    ax = axes[idx]
    subset = df[df['size_category'] == size]
    
    if len(subset) == 0:
        ax.text(0.5, 0.5, f'No {size} lesions', ha='center', va='center', fontsize=14)
        ax.set_title(f'{size.upper()} Lesions (n=0)', fontsize=13, fontweight='bold')
        continue
    
    dice_data = []
    labels_used = []
    colors_used = []
    
    for m, label, color in zip(methods, method_labels, colors):
        col = f'{m}_dice'
        if col in subset.columns and subset[col].notna().sum() > 0:
            dice_data.append(subset[col].dropna().values)
            labels_used.append(label)
            colors_used.append(color)
    
    if dice_data:
        bp = ax.boxplot(dice_data, labels=labels_used, patch_artist=True,
                        medianprops=dict(color='black', linewidth=2))
        for patch, color in zip(bp['boxes'], colors_used):
            patch.set_facecolor(color)
            patch.set_alpha(0.7)
    
    ax.set_title(f'{size.upper()} Lesions (n={len(subset)})', fontsize=13, fontweight='bold')
    ax.set_ylabel('Dice Score')
    ax.tick_params(axis='x', rotation=30)
    ax.set_ylim(0, 1.05)

fig.suptitle('Dice Score Stratified by Lesion Size', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('stratified_dice.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

## 8. Visualize Predictions - Single Case Deep Dive

In [ ]:
import matplotlib.patches as mpatches

def normalize_for_display(img, pct_low=1, pct_high=99):
    """Normalize image to [0,1] for display."""
    lo = np.percentile(img, pct_low)
    hi = np.percentile(img, pct_high)
    if hi - lo > 0:
        return np.clip((img - lo) / (hi - lo), 0, 1)
    return np.zeros_like(img)

def find_best_slice(volume_3d):
    """Find the axial slice with the most lesion content."""
    if volume_3d.ndim != 3:
        return 0
    sums = [np.sum(volume_3d[:, :, i] > 0) for i in range(volume_3d.shape[2])]
    return int(np.argmax(sums)) if max(sums) > 0 else volume_3d.shape[2] // 2

def overlay_mask(mri_slice, mask_slice, color=(1, 0, 0), alpha=0.4):
    """Overlay a binary mask on a grayscale MRI slice."""
    mri_norm = normalize_for_display(mri_slice)
    rgb = np.stack([mri_norm]*3, axis=-1)
    mask = mask_slice > 0
    if np.any(mask):
        for c in range(3):
            rgb[mask, c] = rgb[mask, c] * (1 - alpha) + color[c] * alpha
    return np.clip(rgb, 0, 1)

# ─── Pick a case (choose from results) ───
# Sort by fusion dice to pick an interesting case
if 'learned_fusion_dice' in df.columns:
    median_case = df.iloc[(df['learned_fusion_dice'] - df['learned_fusion_dice'].median()).abs().argsort().iloc[0]]
else:
    median_case = df.iloc[len(df)//2]

case_id = median_case['case_id']
print(f"Visualizing case: {case_id}")

data = dict(np.load(fusion_data_dir / f'{case_id}.npz'))
label = data['label']
image = data['image']  # [3, H, W, D]

# Find best slice
best_z = find_best_slice(label)
print(f"Best slice (most lesion): z={best_z}")
print(f"Lesion volume: {label.sum():.0f} voxels")

In [ ]:
# ─── Multi-panel visualization for the selected case ───
z = best_z

fig, axes = plt.subplots(2, 4, figsize=(22, 10))

# Row 1: Input modalities + Ground Truth
modality_names = ['DWI', 'ADC', 'FLAIR']
for i, name in enumerate(modality_names):
    ax = axes[0, i]
    ax.imshow(np.rot90(normalize_for_display(image[i, :, :, z])), cmap='gray')
    ax.set_title(name, fontsize=13, fontweight='bold')
    ax.axis('off')

# Ground truth overlay on DWI
ax = axes[0, 3]
gt_overlay = overlay_mask(image[0, :, :, z], label[:, :, z], color=(0, 1, 0), alpha=0.5)
ax.imshow(np.rot90(gt_overlay))
ax.set_title('Ground Truth', fontsize=13, fontweight='bold', color='green')
ax.axis('off')

# Row 2: Model predictions
model_info = [
    ('nnunet', 'nnU-Net', '#e74c3c'),
    ('swin_unetr', 'Swin-UNETR', '#3498db'),
    ('segresnet', 'SegResNet', '#2ecc71'),
]

for i, (key, name, hex_color) in enumerate(model_info):
    ax = axes[1, i]
    if key in data:
        pred = data[key]
        pred_bin = (pred > 0.5).astype(np.uint8)
        
        # Overlay: red=prediction, green=GT, yellow=overlap
        mri_norm = normalize_for_display(image[0, :, :, z])
        rgb = np.stack([mri_norm]*3, axis=-1)
        
        pred_mask = pred_bin[:, :, z] > 0
        gt_mask = label[:, :, z] > 0
        overlap = pred_mask & gt_mask
        only_pred = pred_mask & ~gt_mask
        only_gt = gt_mask & ~pred_mask
        
        alpha = 0.5
        # True Positive (yellow)
        rgb[overlap, 0] = rgb[overlap, 0] * (1-alpha) + 1.0 * alpha
        rgb[overlap, 1] = rgb[overlap, 1] * (1-alpha) + 1.0 * alpha
        rgb[overlap, 2] = rgb[overlap, 2] * (1-alpha) + 0.0 * alpha
        # False Positive (red)
        rgb[only_pred, 0] = rgb[only_pred, 0] * (1-alpha) + 1.0 * alpha
        rgb[only_pred, 1] = rgb[only_pred, 1] * (1-alpha) + 0.2 * alpha
        rgb[only_pred, 2] = rgb[only_pred, 2] * (1-alpha) + 0.2 * alpha
        # False Negative (green)
        rgb[only_gt, 0] = rgb[only_gt, 0] * (1-alpha) + 0.2 * alpha
        rgb[only_gt, 1] = rgb[only_gt, 1] * (1-alpha) + 1.0 * alpha
        rgb[only_gt, 2] = rgb[only_gt, 2] * (1-alpha) + 0.2 * alpha
        
        dice_val = compute_metrics(pred_bin, label)['dice']
        ax.imshow(np.rot90(np.clip(rgb, 0, 1)))
        ax.set_title(f'{name}\nDice={dice_val:.3f}', fontsize=12, fontweight='bold')
    else:
        ax.text(0.5, 0.5, 'N/A', ha='center', va='center', fontsize=16)
        ax.set_title(name, fontsize=12)
    ax.axis('off')

# Learned Fusion
ax = axes[1, 3]
if 'image' in data:
    fused_prob = run_fusion(data, fusion_net, device)
    fused_bin = (fused_prob > 0.5).astype(np.uint8)
    
    mri_norm = normalize_for_display(image[0, :, :, z])
    rgb = np.stack([mri_norm]*3, axis=-1)
    
    pred_mask = fused_bin[:, :, z] > 0
    gt_mask = label[:, :, z] > 0
    overlap = pred_mask & gt_mask
    only_pred = pred_mask & ~gt_mask
    only_gt = gt_mask & ~pred_mask
    
    rgb[overlap, 0] = rgb[overlap, 0] * (1-alpha) + 1.0 * alpha
    rgb[overlap, 1] = rgb[overlap, 1] * (1-alpha) + 1.0 * alpha
    rgb[overlap, 2] = rgb[overlap, 2] * (1-alpha) + 0.0 * alpha
    rgb[only_pred, 0] = rgb[only_pred, 0] * (1-alpha) + 1.0 * alpha
    rgb[only_pred, 1] = rgb[only_pred, 1] * (1-alpha) + 0.2 * alpha
    rgb[only_pred, 2] = rgb[only_pred, 2] * (1-alpha) + 0.2 * alpha
    rgb[only_gt, 0] = rgb[only_gt, 0] * (1-alpha) + 0.2 * alpha
    rgb[only_gt, 1] = rgb[only_gt, 1] * (1-alpha) + 1.0 * alpha
    rgb[only_gt, 2] = rgb[only_gt, 2] * (1-alpha) + 0.2 * alpha
    
    dice_val = compute_metrics(fused_bin, label)['dice']
    ax.imshow(np.rot90(np.clip(rgb, 0, 1)))
    ax.set_title(f'Learned Fusion\nDice={dice_val:.3f}', fontsize=12, fontweight='bold', color='purple')
ax.axis('off')

# Legend
legend_elements = [
    mpatches.Patch(facecolor='yellow', alpha=0.8, label='True Positive'),
    mpatches.Patch(facecolor='red', alpha=0.8, label='False Positive'),
    mpatches.Patch(facecolor='green', alpha=0.8, label='False Negative'),
]
fig.legend(handles=legend_elements, loc='lower center', ncol=3, fontsize=12,
           bbox_to_anchor=(0.5, -0.02))

fig.suptitle(f'{case_id} — Slice {z} — All Models Comparison',
             fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'{case_id}_comparison.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

## 9. Probability Heatmap + Uncertainty Visualization

In [ ]:
# ─── Probability heatmaps & uncertainty for the same case ───

fig, axes = plt.subplots(1, 4, figsize=(22, 5))

# Collect probabilities
prob_maps = {}
for key in ['nnunet', 'swin_unetr', 'segresnet']:
    if key in data:
        prob_maps[key] = data[key][:, :, z]

# Individual probability heatmaps
heatmap_info = [
    ('nnunet', 'nnU-Net Probability', 'Reds'),
    ('swin_unetr', 'Swin-UNETR Probability', 'Blues'),
    ('segresnet', 'SegResNet Probability', 'Greens'),
]

for i, (key, title, cmap) in enumerate(heatmap_info):
    ax = axes[i]
    if key in prob_maps:
        im = ax.imshow(np.rot90(prob_maps[key]), cmap=cmap, vmin=0, vmax=1)
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.axis('off')

# Uncertainty map (std across models)
ax = axes[3]
if len(prob_maps) >= 2:
    stacked = np.stack(list(prob_maps.values()), axis=0)
    uncertainty = np.std(stacked, axis=0)
    im = ax.imshow(np.rot90(uncertainty), cmap='hot', vmin=0, vmax=0.5)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    ax.set_title(f'Model Uncertainty (std)\nMax={uncertainty.max():.3f}',
                 fontsize=12, fontweight='bold')
ax.axis('off')

fig.suptitle(f'{case_id} — Slice {z} — Probability Maps & Uncertainty',
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'{case_id}_heatmaps.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

## 10. Multi-Slice Gallery (3x3 Grid)

In [ ]:
# ─── 3x3 grid of slices through the volume ───

total_slices = label.shape[2]
slice_indices = np.linspace(int(total_slices * 0.15), int(total_slices * 0.85), 9, dtype=int)

fig, axes = plt.subplots(3, 3, figsize=(14, 14))

# Run fusion once for the full volume
if 'image' in data:
    fused_prob_full = run_fusion(data, fusion_net, device)
    fused_bin_full = (fused_prob_full > 0.5).astype(np.uint8)
else:
    # Fallback to simple average
    preds = [data[m] for m in ['nnunet', 'swin_unetr', 'segresnet'] if m in data]
    fused_bin_full = (np.mean(preds, axis=0) > 0.5).astype(np.uint8) if preds else np.zeros_like(label)

for idx, sz in enumerate(slice_indices):
    row, col = idx // 3, idx % 3
    ax = axes[row, col]
    
    mri_norm = normalize_for_display(image[0, :, :, sz])
    rgb = np.stack([mri_norm]*3, axis=-1)
    
    pred_mask = fused_bin_full[:, :, sz] > 0
    gt_mask = label[:, :, sz] > 0
    overlap = pred_mask & gt_mask
    only_pred = pred_mask & ~gt_mask
    only_gt = gt_mask & ~pred_mask
    
    a = 0.5
    rgb[overlap, 0] = rgb[overlap, 0]*(1-a) + 1.0*a
    rgb[overlap, 1] = rgb[overlap, 1]*(1-a) + 1.0*a
    rgb[overlap, 2] = rgb[overlap, 2]*(1-a) + 0.0*a
    rgb[only_pred, 0] = rgb[only_pred, 0]*(1-a) + 1.0*a
    rgb[only_pred, 1] = rgb[only_pred, 1]*(1-a) + 0.2*a
    rgb[only_pred, 2] = rgb[only_pred, 2]*(1-a) + 0.2*a
    rgb[only_gt, 0] = rgb[only_gt, 0]*(1-a) + 0.2*a
    rgb[only_gt, 1] = rgb[only_gt, 1]*(1-a) + 1.0*a
    rgb[only_gt, 2] = rgb[only_gt, 2]*(1-a) + 0.2*a
    
    has_lesion = "*" if np.any(gt_mask) else ""
    ax.imshow(np.rot90(np.clip(rgb, 0, 1)))
    ax.set_title(f'Slice {sz} {has_lesion}', fontsize=11)
    ax.axis('off')

legend_elements = [
    mpatches.Patch(facecolor='yellow', alpha=0.8, label='TP (overlap)'),
    mpatches.Patch(facecolor='red', alpha=0.8, label='FP (pred only)'),
    mpatches.Patch(facecolor='green', alpha=0.8, label='FN (GT only)'),
]
fig.legend(handles=legend_elements, loc='lower center', ncol=3, fontsize=11)

fig.suptitle(f'{case_id} — Learned Fusion — Multi-Slice View (* = has lesion)',
             fontsize=15, fontweight='bold')
plt.tight_layout()
plt.subplots_adjust(bottom=0.06)
plt.savefig(f'{case_id}_multislice.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

## 11. Per-Case Dice Score Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# ─── Histogram of Dice scores ───
ax = axes[0]
for m, label_name, color in zip(methods, method_labels, colors):
    col = f'{m}_dice'
    if col in df.columns:
        ax.hist(df[col].dropna(), bins=20, alpha=0.5, label=label_name, color=color, edgecolor='white')
ax.set_xlabel('Dice Score', fontsize=12)
ax.set_ylabel('Number of Cases', fontsize=12)
ax.set_title('Dice Score Distribution', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.set_xlim(0, 1)

# ─── Scatter: Fusion Dice vs Simple Avg Dice ───
ax = axes[1]
if 'learned_fusion_dice' in df.columns and 'simple_avg_dice' in df.columns:
    size_colors = {'small': '#e74c3c', 'medium': '#f39c12', 'large': '#2ecc71'}
    for size in ['small', 'medium', 'large']:
        subset = df[df['size_category'] == size]
        if len(subset) > 0:
            ax.scatter(subset['simple_avg_dice'], subset['learned_fusion_dice'],
                      c=size_colors[size], label=f'{size} (n={len(subset)})',
                      alpha=0.6, s=40, edgecolors='white', linewidth=0.5)
    
    ax.plot([0, 1], [0, 1], 'k--', alpha=0.3, label='y=x (no improvement)')
    ax.set_xlabel('Simple Average Dice', fontsize=12)
    ax.set_ylabel('Learned Fusion Dice', fontsize=12)
    ax.set_title('Fusion Improvement Over Simple Average', fontsize=14, fontweight='bold')
    ax.legend(fontsize=10)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_aspect('equal')
else:
    ax.text(0.5, 0.5, 'Fusion data not available', ha='center', va='center', fontsize=14)

plt.tight_layout()
plt.savefig('dice_distribution.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

## 12. Visualize Any Case (Interactive)

In [ ]:
# ═══════════════════════════════════════════
#  CHANGE THIS to visualize a different case
# ═══════════════════════════════════════════
CASE_ID = 'case_0005'   # <-- change me
SLICE_IDX = None         # None = auto-pick best slice
# ═══════════════════════════════════════════

npz_path = fusion_data_dir / f'{CASE_ID}.npz'
if not npz_path.exists():
    print(f"Case not found: {npz_path}")
    print(f"Available cases: {[f.stem for f in npz_files[:10]]}...")
else:
    d = dict(np.load(npz_path))
    lbl = d['label']
    img = d['image']
    z = SLICE_IDX if SLICE_IDX else find_best_slice(lbl)
    
    fig, axes = plt.subplots(1, 5, figsize=(25, 5))
    
    # DWI
    axes[0].imshow(np.rot90(normalize_for_display(img[0, :, :, z])), cmap='gray')
    axes[0].set_title('DWI', fontsize=13, fontweight='bold')
    axes[0].axis('off')
    
    # Ground truth
    gt_ov = overlay_mask(img[0, :, :, z], lbl[:, :, z], color=(0,1,0), alpha=0.5)
    axes[1].imshow(np.rot90(gt_ov))
    axes[1].set_title('Ground Truth', fontsize=13, fontweight='bold', color='green')
    axes[1].axis('off')
    
    # Each model prediction
    for i, (key, name, clr) in enumerate([('nnunet','nnU-Net',(1,0.3,0.3)),
                                           ('swin_unetr','Swin',(0.3,0.5,1)),
                                           ('segresnet','SegRes',(0.3,0.9,0.3))]):
        ax = axes[i+2]
        if key in d:
            pred_ov = overlay_mask(img[0, :, :, z], (d[key][:, :, z] > 0.5).astype(float), color=clr, alpha=0.5)
            dice_v = compute_metrics((d[key] > 0.5).astype(np.uint8), lbl)['dice']
            ax.imshow(np.rot90(pred_ov))
            ax.set_title(f'{name} (Dice={dice_v:.3f})', fontsize=12, fontweight='bold')
        else:
            ax.text(0.5, 0.5, 'N/A', ha='center', va='center')
        ax.axis('off')
    
    fig.suptitle(f'{CASE_ID} — Slice {z}', fontsize=15, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    print(f"Label volume: {lbl.sum():.0f} voxels | Category: {categorize_lesion_size(lbl.sum())}")

## 13. Export Results

In [ ]:
# Save results CSV
df.to_csv('evaluation_results.csv', index=False)
print(f"Results saved to evaluation_results.csv ({len(df)} cases)")

# Summary stats
print("\n" + "="*60)
print("FINAL SUMMARY")
print("="*60)
for m, label_name in zip(methods, method_labels):
    dice_col = f'{m}_dice'
    if dice_col in df.columns:
        mean_d = df[dice_col].mean()
        std_d = df[dice_col].std()
        print(f"{label_name:<20}: {mean_d:.4f} +/- {std_d:.4f}")
print("="*60)

# Download files from Colab
try:
    from google.colab import files
    files.download('evaluation_results.csv')
    files.download('metrics_comparison.png')
    print("\nFiles downloaded!")
except:
    print("\nFiles saved locally (not in Colab environment).")